In [1]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv

import pyinputplus as pyip
import pprint
import json

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime # can provide a description or name of function
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
data_path = "../data/short_PUDF_base1_1q2019_tab.csv"

# Load agent
model = init_chat_model(
    "MiniMaxAI/MiniMax-M2.5",
    model_provider="openai",
    base_url=os.environ.get("BASE_URL"),
)

# can implement a checkpointer to add memory to the model
agent = create_agent(model=model,)

/home/alewis/Downloads/agentic-synth-data/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [12]:
def select_preferences() -> dict:
    cols_of_interest = [] # if empty, all columns should be included
    synthesis_request = {
        # Q: What do you want to use the synthetic data for?
        # Q: If applicable, what is the target column?
        "synthesis_goal": {
            # goal: basic_generation, predictive_utility
            "goal": "basic_generation",
            # Only for predictive_utility: regression, classification, survival, None
            "task": None,
            "target_column": None,
        },
        # Q: Which columns should be included?
        "data_scope": {
            "selected_columns": cols_of_interest # If empty, all cols
            },
        # Q: How many synthetic samples should be generated?
        "sample_size": {
            # mode: fixed, fraction_of_original, same_as_original
            "mode": "fixed",
            "value": 10, # just for testing workflow
        },
        # Q: Should privacy preserving synthesis be used?
        "privacy": {
            # standard, differentially private
            "mode": "standard",
            "sensitive_columns": [],
        },
        # Possibly preprocessing where missing value strategy and dtype correction can be done?
        "generator": {
            "name": "auto", # Recommend data generator
            #"parameters": {}, # Depends on generators used/available
        },
        "missing_value_handling": {
            # mode: impute, threshold, remove
            "mode": "auto",
            "imputation_method": None, # mean, median, SMOTE?
            "set_threshold": None, # minimum number of missing columns in a sample to disregard
        }
        # Possibly a note for the evaluation type as well?
    }

    # Fill synthesis goal
    print("What do you want to use the synthetic data for?")

    choice = pyip.inputMenu(["basic generation", "predictive utility"], numbered=True)
    # Q: What do you want to use the synthetic data for?
    if choice == "predictive utility":
        synthesis_request["synthesis_goal"]["goal"] = "predictive_utility"
        print("Which downstream machine learning task are you interested in?")
        
        choice = pyip.inputMenu(["classification", "regression", "survival"], numbered=True) # default none
        synthesis_request["synthesis_goal"]["task"] = choice
        print("What is the target column? If you'd like suggestions, enter 'auto'.")
        # TODO think about how to include column names in an easier way
        #target = pyip.inputChoice([],"If applicable, what is the target column?") # one entry or auto?
        #synthesis_request["synthesis_goal"]["target_column"] = target 

    # Fill data_scope
    #response = pyip.inputYesNo("Are there any columns in the dataset that you would like to exclude from generation?")
        # Q: Which columns should be included?
        # TODO add above questoin as prompt
    synthesis_request["data_scope"]["selected_columns"] = cols_of_interest

    # Fill sample size
    print("Would you like to generate the same amount of samples as the dataset or a fixed number of samples?")

    choice = pyip.inputMenu(["fixed value", "same as original"], numbered=True) # fixed, same_as_original, fraction_of_original
    synthesis_request["sample_size"]["mode"] = choice
    
    if choice == "fixed value":
        print("How many synthetic samples would you like to generate?")

        value = pyip.inputInt(min=1)
        synthesis_request["sample_size"]["value"] = value

    # elif choice == "fraction": # could probably leave this out for now.
    #     value = pyip.inputFloat()
        # TODO incorporate as a fraction (nearest whole number)

    # Fill privacy
    response = pyip.inputYesNo("Are you interested in differential privacy?")
    #response = pyip.inputChoice(["standard", "differentially private"])
    if response == "yes":
        synthesis_request["privacy"]["mode"] = "differential_privacy"
        #pyip.inputYesNo()
        # TODO: check if there are any sensitive columns (what is a way to include columns that is not cumbesome?)

    # Fill data generator preferences
    response = pyip.inputYesNo("\nIs there a specific data generator that you want to use on your data?")
    if response == "yes":
        # show what is available.
        # pyip.inputMenu(["CTGAN", "PATEGAN", "Basian Network"]) and so on...
        pass
    else:
        print("\n\tYou will receive a recommendation based on the dataset's characteristics.")
        # check that the data generator is available for use (probably ctgan, pategan, and tvae)
        # check parameter preferences of the user. Depends on generators used/available
        
    # Missing value handling preferences
    print("How would you like missing values to be handled? Enter 'auto' if you would like a suggestion.")
    choice = pyip.inputMenu(["impute", "threshold", "remove", "auto"], numbered=True)
    if choice == "auto":
        synthesis_request["missing_value_handling"]["mode"] = choice

    elif choice == "threshold":
        synthesis_request["missing_value_handling"]["mode"] = choice
        value = pyip.inputInt("What is the minimum number of missing columns in a sample remove from the data?")
        synthesis_request["missing_value_handling"]["set_threshold"] = value

    elif choice == "remove":
        synthesis_request["missing_value_handling"]["mode"] = choice

    elif choice == "impute":
        synthesis_request["missing_value_handling"]["mode"] = choice
        pass
        # imputation method
        # still have to consider how categorical infomation should be filled in.
    
    print("Your preferences have been recorded:")
    pprint.pprint(synthesis_request)
    return synthesis_request

synthesis_request = select_preferences()

What do you want to use the synthetic data for?
Please select one of the following:
1. basic generation
2. predictive utility


Would you like to generate the same amount of samples as the dataset or a fixed number of samples?
Please select one of the following:
1. fixed value
2. same as original
How many synthetic samples would you like to generate?
Are you interested in differential privacy?
Is there a specific data generator that you want to use on your data?'yep' is not a valid yes/no response.

Is there a specific data generator that you want to use on your data?How would you like missing values to be handled? Enter 'auto' if you would like a suggestion.
Please select one of the following:
1. impute
2. threshold
3. remove
4. auto
Your preferences have been recorded:
{'data_scope': {'selected_columns': []},
 'generator': {'name': 'auto'},
 'missing_value_handling': {'imputation_method': None,
                            'mode': 'remove',
                            'set_threshold': None},
 'privacy': {'mode': 'standard', 'sensitive_columns': []},
 'sample_size': {'mode': 'fixed value', 'value': 10},
 'synth

In [3]:
pprint.pprint(synthesis_request)

{'data_scope': {'selected_columns': []},
 'generator': {'name': 'auto'},
 'missing_value_handling': {'imputation_method': None,
                            'mode': 'remove',
                            'set_threshold': None},
 'privacy': {'mode': 'standard', 'sensitive_columns': []},
 'sample_size': {'mode': 'fixed value', 'value': 10},
 'synthesis_goal': {'goal': 'basic_generation',
                    'target_column': None,
                    'task': None}}


In [13]:
def upload_data(upload_path:str, selected_columns:list) -> pd.DataFrame:
    """
    Verify uploaded data can be made into a pandas dataframe.
    Currently only tested for one file, would need to be updated to
    account for multiple files (synthcraft has a multiple file uploader
    which might be worth looking into)
    """
    path = upload_path.lower()

    # if path.endswith(".xlsx"):
    # If this shoudl be added, needs some for cols that are not selected
    #     df = pd.read_excel(upload_path, usecols=selected_columns)
    #     if df.shape[0] > 1 and df.shape[1] > 1:
    #         raise ValueError("Invalid entry. Verify excel file is in a tidy format (example: https://aeturrell.github.io/python4DS/data-tidy.html).")
    #     return df

    for sep in [",", "\t", ";", "|"]:
        try:
            if len(selected_columns) == 0:
                df = pd.read_csv(upload_path, delimiter=sep)
                if df.shape[0] > 1 and df.shape[1] > 1:
                    return df
            else:
                df = pd.read_csv(upload_path, usecols=selected_columns, delimiter=sep)
                if df.shape[0] > 1 and df.shape[1] > 1:
                    return df
            
        except Exception:
            continue

    raise ValueError("Invalid entry; File type may not be supported. Verify file is in a tidy format (example: https://aeturrell.github.io/python4DS/data-tidy.html).")


df = upload_data(data_path, synthesis_request["data_scope"]["selected_columns"])
df = df.drop(columns=["PRINC_DIAG_CODE", "ADMITTING_DIAGNOSIS"]) # high cardinality

/tmp/ipykernel_815218/1950129129.py:20: DtypeWarning: Columns (0: RACE, 1: ETHNICITY, 2: PAT_AGE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(upload_path, delimiter=sep)


In [5]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 774564 entries, 0 to 774563
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   RECORD_ID             774564 non-null  int64  
 1   TYPE_OF_ADMISSION     774563 non-null  float64
 2   SOURCE_OF_ADMISSION   774541 non-null  str    
 3   PUBLIC_HEALTH_REGION  717419 non-null  float64
 4   SEX_CODE              712721 non-null  str    
 5   RACE                  774541 non-null  object 
 6   ETHNICITY             771952 non-null  object 
 7   ADMIT_WEEKDAY         774559 non-null  float64
 8   LENGTH_OF_STAY        774559 non-null  float64
 9   PAT_AGE               774564 non-null  object 
 10  FIRST_PAYMENT_SRC     774481 non-null  str    
 11  POA_PRINC_DIAG_CODE   641129 non-null  str    
dtypes: float64(4), int64(1), object(3), str(4)
memory usage: 260.0 MB


In [7]:
def handle_dtypes(df:pd.DataFrame):
    # TODO print data types to the user and confirm if valid.
    # TODO figure out if it woudl be helpful to state categorical numerical (and greater details here)
    object_cols = df.select_dtypes(include="object").columns
    mixed_type_cols = []

    for col in object_cols:
        types = df[col].dropna().map(type).unique()
        if len(types) > 1:
            mixed_type_cols.append((col, types))

    print(f"There are {len(mixed_type_cols)} columns with mixed types {mixed_type_cols}")

    # Possibly easier to check here for missing values denoted as ? or other characters
    # probably a better way to represent this too. the data tpes
    try:
        for col, types in mixed_type_cols:#in range(len(mixed_type_cols)):
            choice = input(f"The `{col}` column has the following data types: {types}\nConvert to (numeric/categorical/datetime/skip):")
            if choice == "numeric":
                df[col] = pd.to_numeric(df[col])
            elif choice == "categorical":
                df[col] = df[col].astype(str)
                # Or as type category (but have to see if the geneators are happy with that...)
            elif choice == "datetime":
                df[col] = pd.to_datetime(df[col])
        return df, df.dtypes
    except:
        # maybe this part is a bit too dynamic and would function better with help of an llm
        # Needs to be fixed.
        Exception("Retyping failed. Are there any missing values denoted by a particular symbol in the data (e.g., '`', '?'?")
        character = input("Enter a character that represents missing values.")
        # Are missing or invalid value entries referenced by a particular string (e.g. '`', '?')?
        df = df.copy().replace(character, np.nan)
        # would it be helpful to include an llm in this part?
        return handle_dtypes(df)
    
df, dtypes = handle_dtypes(df)

/tmp/ipykernel_815218/1996870075.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include="object").columns


There are 0 columns with mixed types []


---
If auto is selected (example):

In [8]:
# temporary to show agent output
proportion_missing = (df.isna().sum() / df.shape[0]).round(4).to_dict()
rows_with_missing = int(df.isna().any(axis=1).sum())

prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are advising on missing value handling for synthetic tabular data preprocessing.

Always return exactly one valid JSON object.
No markdown.
No text outside the JSON.

The response must include:
- a recommended action
- brief rationale

Valid modes:
- remove_all
- threshold
- impute

JSON format:
{{
"missing_value_handling": {{
    "mode": "remove | threshold | impute",
    "set_threshold": null,
    "columns_to_review": [],
    "rationale": "Brief explanation of why this plan is recommended."
}}
}}
"""),
    ("user", """
Missing value proportions per column:
{proportion_missing}

Rows with at least one missing value:
{rows_with_missing}

Total rows:
{total_rows}

Current synthesis request:
{synthesis_request}
""")
])

chain = prompt | agent
plan = chain.invoke({
    "proportion_missing": proportion_missing,
    "rows_with_missing": rows_with_missing,
    "total_rows": df.shape[0],
    "synthesis_request": synthesis_request,
    })
plan = plan['messages'][-1].content.split("</think>")[1].replace("\n", "")
print(f"Plan: {plan}")

parsed = json.loads(plan)

missing_plan = parsed["missing_value_handling"]

print("Recommended mode:", missing_plan["mode"])
print("Threshold:", missing_plan["set_threshold"])
print("Columns to review:", missing_plan["columns_to_review"])
print("Rationale:", missing_plan["rationale"])
# A similar thing can be done for the generator recommendation

Plan: {"missing_value_handling":{"mode":"impute","set_threshold":null,"columns_to_review":["PUBLIC_HEALTH_REGION","SEX_CODE","POA_PRINC_DIAG_CODE","ETHNICITY","FIRST_PAYMENT_SRC"],"rationale":"Impute is recommended over remove because 220704 rows (28.5% of data) would be discarded with removal. POA_PRENC_DIAG_CODE has the highest missing rate at 17.23%, followed by SEX_CODE (7.98%) and PUBLIC_HEALTH_REGION (7.38%). Imputation preserves data volume and statistical properties while handling missing values appropriately for synthetic data generation."}}
Recommended mode: impute
Threshold: None
Columns to review: ['PUBLIC_HEALTH_REGION', 'SEX_CODE', 'POA_PRINC_DIAG_CODE', 'ETHNICITY', 'FIRST_PAYMENT_SRC']
Rationale: Impute is recommended over remove because 220704 rows (28.5% of data) would be discarded with removal. POA_PRENC_DIAG_CODE has the highest missing rate at 17.23%, followed by SEX_CODE (7.98%) and PUBLIC_HEALTH_REGION (7.38%). Imputation preserves data volume and statistical pro

---

In [9]:
def update_missing_value_choice(synthesis_request: dict):
    print("Please select another method to handle missing values:")

    choice = pyip.inputChoice(["impute", "threshold", "auto"])
    synthesis_request["missing_value_handling"]["mode"] = choice

    # If any additional specifications are required
    if choice == "threshold":
        value = pyip.inputInt("Set the maximum number of missing values allowed per row:", min=0)
        synthesis_request["missing_value_handling"]["set_threshold"] = value

    elif choice == "impute":
        method = pyip.inputChoice(["NA"], prompt="Please select an imputation method:")
        synthesis_request["missing_value_handling"]["imputation_method"] = method
        # TODO: add additional specifications

    return synthesis_request


def handle_missing_values(df:pd.DataFrame, synthesis_request:dict):
    #TODO add an initial check for missing values
    mode = synthesis_request["missing_value_handling"]["mode"]
    # 1. Remove all samples with NA columns
    if mode == "remove":
        null_row_count = df.isna().any(axis=1).sum()
        if null_row_count/df.shape[0] >= .4: # not sure where to set or if to always warn
            print(f"Warning: {null_row_count} out of {df.shape[0]} rows ({null_row_count/df.shape[0]*100}%) will be removed from the dataset.")
            confirm = pyip.inputYesNo("Confirm whether this should be done (yes/no/y/n):")
            if confirm == "yes":
                df = df.dropna()
                return df, synthesis_request
            else:
                # If this option is too harsh and the user wants to try another method.
                synthesis_request = update_missing_value_choice(synthesis_request)
                return handle_missing_values(df, synthesis_request)
        else:
            df = df.dropna()
            return df, synthesis_request
        
    # 2. Remove all samples above a specified threshold of NA columns 
    if mode == "threshold":
        # possible TODO allow for threshold changes (if found to be too harsh but want to stay with a threshold)
        threshold = synthesis_request["missing_value_handling"]["set_threshold"]
        # TODO add a limit to the threshold such that it doesn't add mroe
        threshold_samples = df.isna().sum(axis=1) > threshold
        count = threshold_samples.sum()

        # Maybe just warn generally and ask for confirmation from user?
        if count/df.shape[0] >= .4: # not sure where exactly to set the limit
            print(f"Warning: {count} out of {df.shape[0]} ({count/df.shape[0]*100}%) will be removed from the dataset.")
            confirm = pyip.inputYesNo("Confirm whether this should be done.")
            if confirm == "yes":
                df = df[~threshold_samples]
                return df, synthesis_request
            else:
                synthesis_request = update_missing_value_choice(synthesis_request)
                return handle_missing_values(df, synthesis_request)
        else:
            df = df[~threshold_samples]
            return df, synthesis_request

    # TODO Look into how categorical columns can be imputed (mixed data types)
    # 3. Impute the data with the mean. 
    # but what about categorical columns? 
    # See about mode and possibly knn imputation (but might be costly)
    # Feature engine? look into.
    if mode == "mean":
        pass

    # 4. Impute the data with the median
    if mode == "median":
        pass

    # 5. Use an LLM to suggest the missing value handling strategy
    if mode == "auto":
        # check how many missing values there are in each column compared to the total
       # Quick example prompt for recommendation (would have to update)
       # TODO add addtional options as options
        # 3. Impute the mean (mean).
        # 4. Impute the median (median).
        # 5. Impute with another method (other).
        proportion_missing = (df.isna().sum() / df.shape[0]).round(4).to_dict()
        rows_with_missing = int(df.isna().any(axis=1).sum())

        prompt = ChatPromptTemplate.from_messages([
            ("system", """
        You are advising on missing value handling for synthetic tabular data preprocessing.

        Always return exactly one valid JSON object.
        No markdown.
        No text outside the JSON.

        The response must include:
        - a recommended action
        - brief rationale

        Valid modes:
        - remove_all
        - threshold
        - impute

        JSON format:
        {{
        "missing_value_handling": {{
            "mode": "remove | threshold | impute",
            "set_threshold": null,
            "columns_to_review": [],
            "rationale": "Brief explanation of why this plan is recommended."
        }}
        }}
        """),
            ("user", """
        Missing value proportions per column:
        {proportion_missing}

        Rows with at least one missing value:
        {rows_with_missing}

        Total rows:
        {total_rows}

        Current synthesis request:
        {synthesis_request}
        """)
        ])
        
        chain = prompt | agent
        plan = chain.invoke({
            "proportion_missing": proportion_missing,
            "rows_with_missing": rows_with_missing,
            "total_rows": df.shape[0],
            "synthesis_request": synthesis_request,
            })
        plan = plan['messages'][-1].content.split("</think>")[1].replace("\n", "")

        parsed = json.loads(plan)

        missing_plan = parsed["missing_value_handling"]

        print("Recommended mode:", missing_plan["mode"])
        print("Threshold:", missing_plan["set_threshold"])
        print("Columns to review:", missing_plan["columns_to_review"])
        print("Rationale:", missing_plan["rationale"])
        # can be used to update the details in synthesis request
        choice = pyip.inputYesNo("Would you like to proceed with the suggestion?")
        # TODO add change if agreement
        # TODO add a way for user to include more information at disagreement
        pass

df, synthesis_request = handle_missing_values(df, synthesis_request)

In [10]:
# Save cleaned dataframe
df.to_csv("../data/clean_short_PUDF_base1_1q2019_tab.csv", index=False)

In [11]:
# make json for dataframe
csv_path = "../data/clean_short_PUDF_base1_1q2019_tab.csv"
json_path = "../data/clean_short_PUDF_base1_1q2019_tab.json"
categorical_cols = [
    "TYPE_OF_ADMISSION",
    "SOURCE_OF_ADMISSION",
    "PUBLIC_HEALTH_REGION",
    "SEX_CODE",
    "RACE",
    "ETHNICITY",
    "ADMIT_WEEKDAY",
    "PAT_AGE",
    "FIRST_PAYMENT_SRC",
    "POA_PRINC_DIAG_CODE",
]

continuous_cols = ["LENGTH_OF_STAY"]

dtype_map = {col: str for col in categorical_cols}
df = pd.read_csv(csv_path, dtype=dtype_map)

columns = []

for col in df.columns:
    if col in categorical_cols:
        values = sorted(df[col].dropna().astype(str).unique().tolist())
        columns.append({
            "name": col,
            "type": "Categorical",
            "i2s": values,
            "size": len(values)
        })

    elif col in continuous_cols:
        columns.append({
            "name": col,
            "type": "Float",
            "min": float(df[col].min()),
            "max": float(df[col].max())
        })

metadata = {"columns": columns}

with open(json_path, "w") as f:
    json.dump(metadata, f, indent=2)

- Numerical: CTGAN
  - Privacy: PATEGAN
- Categorical: BaysianNet
- Mix: CTGAN

In terminal:
can change tool used, number of synthetic samples and so on.
```bash
podman run -it --rm   --entrypoint /bin/bash   -v "$PWD:/workspace:Z"   docker.io/springepfl/synthetic-data:latest   -c "cd /synthetic_data_release && python3 executables/generate_synthetic_dataset.py \
  -D /workspace/data/clean_short_PUDF_base1_1q2019_tab \
  -M PATEGAN \
  -O /workspace/data/synthetic_pategan.csv \
  -N 10"

To set up podman:
Note: output instead of data. 

```
podman run -it --rm -v "$(pwd)/data:/data" -p 8888:8888 docker.io/springepfl/synthetic-data:latest